In [3]:
import cv2
import time

IDX = 1  # 예전에 잡힌 인덱스로 먼저 테스트 추천!

for backend in [cv2.CAP_DSHOW, cv2.CAP_MSMF]:
    print("backend:", backend)

    cap = cv2.VideoCapture(IDX, backend)
    print("isOpened:", cap.isOpened())

    # 워밍업(최대 2초)
    t0 = time.time()
    ok = False
    while time.time() - t0 < 2.0:
        ret, frame = cap.read()
        if ret and frame is not None:
            ok = True
            break

    if not ok:
        print("FAIL: no frame")
        cap.release()
        continue

    print("SUCCESS: streaming")
    while True:
        ret, frame = cap.read()
        if not ret or frame is None:
            print("stream lost")
            break
        cv2.imshow(f"test backend {backend}", frame)
        if cv2.waitKey(1) == 27:  # ESC
            break

    cap.release()
    cv2.destroyAllWindows()

backend: 700
isOpened: True
SUCCESS: streaming
backend: 1400
isOpened: True
SUCCESS: streaming


In [6]:
import cv2
from ultralytics import RTDETR

# 모델 로드
model = RTDETR("rtdetr-l.pt")

# 캡처 열기
cap = cv2.VideoCapture(1, cv2.CAP_DSHOW)

# 워밍업
import time
t0 = time.time()
while time.time() - t0 < 2.0:
    ret, frame = cap.read()
    if ret and frame is not None:
        break

while True:
    ret, frame = cap.read()
    if not ret or frame is None:
        continue

    results = model.predict(
        source=frame,
        imgsz=640,
        conf=0.35,
        # classes=[2, 5, 7],  # car, bus, truck
        verbose=False,
    )

    # 박스 그리기
    annotated = results[0].plot()

    cv2.imshow("RTDETR Live", annotated)
    if cv2.waitKey(1) == 27:  # ESC
        break

cap.release()
cv2.destroyAllWindows()